# 006 · Results-day move model

**The question.** When a company publishes its quarterly or annual results, how likely is its
share price to jump or fall sharply, beyond what the whole market does over the same days?

**The answer.** `move`: the size of the price change, up or down, from the close before the
results were released (`as_of`) to the close after the results filing (the 8-K, `close_after`),
minus the market's (SPY) change over the same days. One row per results release. The forecast is
**three chances**: of a move above 5%, above 10% and above 20% (D62), shown in five bands with
their past hit rates, never as one bare number.

**The baseline to beat** (pre-registered): `prev4_mean`, the mean of the company's previous four
results-day moves (`n_prev` says how many earlier moves it had; none for a company's first
release).

**How the levels are judged** (V2-41): the mean log loss of the three chances; rank correlation,
absolute and log errors beside it. The simplest level within one standard error of the yearly
gaps to the best is chosen (V2-50).

**Status.** Rebuilt on 9 November 2026 ([spec](../specs/006-results-move-model/spec.md),
[plan](../specs/006-results-move-model/plan.md),
[research](../specs/006-results-move-model/research.md), issue #77). Gemini's first version
(30 Oct – 3 Nov) used the look-ahead input `cum_split_after` and is void (audit, 4 Nov).
**Approved by the researcher on 9 November (V2-55): LightGBM.** Its code is
`src/sentinel/models/results_move.py`, which this notebook imports.

### What the answer means (read this first)

Think of an exam: the **features** are the questions (what was known about a company at the close
before its results: how much its price usually moves, its size, its finances, recent filings, the
market's mood, and its own earlier results-day moves), and **`move`** is the answer key (what
happened). The model learns from releases *with* the key and is judged on releases whose key it
never saw.

- **The answer, with an example.** A company's shares usually move about 1.5% a day. It releases
  its results before the market opens; by the close after its 8-K filing the shares are down 11%
  while the market (SPY) is up 1%. Its move is |−11% − 1%| = **12%**: above 5% yes, above 10% yes,
  above 20% no. Subtracting the market keeps only what the results did, not what every stock did
  that day.
- **Why chances, not one number.** Nobody can forecast the size of one release's move exactly; a
  chance is honest about that. "A 30% chance of a move above 10%" means: of releases given 30%,
  about three in ten really moved more than 10%. Section 8 checks exactly that.
- **How a chance is scored: log loss.** If the model says 30% and the move happens, it is charged
  −log(0.30) = 1.20; if it does not happen, −log(0.70) = 0.36. Confident and right costs little,
  confident and wrong costs a lot. The average over all releases and the three events is the
  deciding measure (V2-41): lower is better. Always giving every release the same chances (the
  base rates: about 40%, 17% and 4%) scores 0.43–0.51 on the test years; this is the floor every
  level must beat.
- **Which rows and columns.** Fit only on `results_move.parquet`, releases with `eligible` true
  and `move` present; the inputs are the export's features at the close before the release,
  without the three release-timing features (V2-54), plus `prev4_mean` and `n_prev`.
- **How error is measured, in three stages, always on answers the model has not seen:** (1) the
  walk-forward test years 2019–2022 here (learn from earlier years, score the year: this chooses
  the model); (2) the holdout, once (feature 010): the guarded command computes the 2023+ releases
  and their moves and scores the frozen model; (3) live (feature 009, after 010): each day's
  forecast is stored and scored after the company's next release.

### Rules for every model notebook

- **Spec first.** The spec, plan and tasks are in `specs/006-results-move-model/` (rewritten on
  9 Nov), on branch `006-results-move-model`.
- **This notebook is where the model is built.** Every level is created, run and tuned here,
  with its tables and charts, start to finish by one command, so the researcher can read how the
  choice was made. Once the researcher approves the chosen model, its code moves into
  `src/sentinel/models/results_move.py` and this notebook imports it and must give the same
  numbers (docs/plan.md, "Notebooks and code").
- **Data.** Read only `data/exports/ml/results_move.parquet` through
  `sentinel.eda.load("results_move")`, which keeps the development rows (dated 2022 or earlier).
  Learn and test only on releases with `eligible` true and `move` present. Never read
  `data/exports/reference/` and never compute anything for 2023 or later: that is the sealed
  holdout, tested once by feature 010 (constitution IV).
- **Never use as inputs:** `move`, `close_after`, `filing_date`, `report_date`, `cik`, `as_of`, the
  flags, `sector` (each company's *current* industry code, not point-in-time), `cum_split_after`
  (the split ratios after the row's day: look-ahead) and the three release-timing features
  `days_since_results`, `days_to_next_results`, `months_since_report` (V2-54: on any day but a
  release eve they mean something else, and the app scores every day). `prev4_mean` and `n_prev`
  are declared inputs (spec FR-002).
- **Never a random train/test split.** Releases of one company over time are not independent, and
  a random split lets the model learn from the future. The only split is walk-forward by date.
- **Evaluation is fixed** (D60, `config/risk_params.yaml` v5, never edited): test years 2019,
  2020, 2021, 2022; four levels: naive baseline → rule → classic model → LightGBM; the **simplest
  level within one standard error of the best** (V2-50: the standard error of the four yearly
  differences from the best level); every test year reported, not only the average.
- **Every fitted step knows when it learned (V2-49).** Each forecast model, calibration and set
  of band edges is a `sentinel.models.dated.Dated`: it learns only from releases whose moves were
  known before its cutoff and refuses to forecast releases dated before it. No early stopping:
  the number of trees is chosen by predicting the validation year. The folds come from
  `eda.folds` and are checked by `eda.check_folds`.
- **The method's choices were made before any test year** (research.md, design check on
  2016–2018): one forecast of the log move per level, turned into the three chances by
  calibration; training from 2010; no sector; fewer trees.
- **Reproducible:** rows sorted by `cik`, `as_of`; fixed seeds (`SEED = 0`); each run logged in
  MLflow (experiment `006-results-move`) with the git commit, the export date and a data hash.
  Two runs must give identical predictions.
- **Private data:** outputs are stripped on commit (nbstripout); written conclusions quote
  aggregates only, never one company's prices or returns (Tiingo terms).
- **Run it** start to finish under the memory cap:
  `systemd-run --user --scope -p MemoryMax=5G -p MemorySwapMax=0 uv run jupyter nbconvert --to notebook --execute --inplace notebooks/006-results-move-model.ipynb`
  (with `SENTINEL_TRIAL=1` it runs one setting and one seed: a check of the code, not results).

### Two kinds of test, and the final one is not in this notebook

1. **Test years 2019–2022 (here).** For each test year, learn from the earlier years only and
   score that year. This is where the four levels are compared and **one is chosen**. Because the
   choice is made by looking at these results, they flatter the winner a little.
2. **The holdout: releases from 2023 on (not here, not in any notebook).** It answers the only
   question that matters in the end: does the model chosen here work on data nobody looked at
   while choosing it? That works only if it is used **once**.

**How it stays hidden.** The training file has no moves after 2022: the pipeline does not compute
them (the label code stops at windows ending 31 December 2022). This notebook does not read the
private reference copies in `data/exports/reference/` (feature 002).

**How the final test is run (feature 010, 16 November).** This feature ends with the chosen model
**frozen** in `src/sentinel/models/` (settings, training rows to 2022, seed, git commit, export
date, logged in MLflow). The researcher then runs **one guarded command** (010): it checks that the
models are frozen and that no holdout run exists yet, computes the 2023+ releases and their moves
(and with them `prev4_mean` and `n_prev`), scores the frozen models and writes the results with a
log entry; a second run is refused. The results go into the report as they came out; the model is
not changed afterwards (constitution IV).

### Make the model holdout-ready

1. **Inputs selected by name**: the feature list is saved with the model, rows are read as
   `rows[features]`.
2. **Every preprocessing step lives inside the model** (logs, cut points, fill values, missing
   flags, scaling), fitted on the training releases only.
3. **One object**: the forecast, the three calibrations, the band edges (each a `Dated`), the
   feature list, the eligibility flag (`eligible`), the seed, the git commit and the export date.
4. **Dry run before 16 November** (section 10): score the 2023+ release eves of
   `feature_snapshot.parquet` and `latest.parquet` (features only, no answers): one forecast per
   eligible row, chances in order. This looks at no outcome, so it does not spend the holdout.
5. **Scoring is one function** `predict(model, rows)`, the same for the test years, the dry run,
   the holdout and the daily scores.

### What the EDA found for this model (notebooks/004-eda-results-move.ipynb, Findings)

- About 157,000 releases (5,900 companies) are eligible with an answer, 2010–2022; no foreign
  filers (they file no 8-K item 2.02). Median move 3.7% beyond the market; 99% below 31%; a
  handful exactly 0.
- The baseline **ranks** releases (rank correlation 0.30–0.39 in 2019–2022) but its size is no
  more accurate than one constant for everyone: judge by ranking and by the chances.
- **Volatility ranks the move as well as the company's own history** (ρ ≈ 0.33 for
  `idio_vol_252`, `range_vol_21`, `vol_252`, `vol_63`): combine the two. Then the fall from the
  12-month high, beta, loss-making, cash share, price level, size.
- Model the log of the move: the tail is long.
- Financial features are mostly missing in 2010–2011 (small companies started tagging their
  reports in 2011): consider training from 2012 (checked in section 9; the design check on
  2016–2018 kept 2010).
- 1.9% of releases have an 8-K filed 5 or more days after the release, with larger moves (other
  news in the window): the test years are shown with and without them (section 9).
- No leakage: every row is dated before its release; each earlier move used by `prev4_mean`
  ended before the row's date (checked again on 9 Nov: 0 of 170,879 rows overlap).

**Data cleaning** (V2-33): the pipeline's cleaning is frozen; any further cleaning is part of this
model (logs, cuts at the 1st and 99th percentiles, "was missing" flags, fitted on training
releases only) and moves into `src/sentinel/models/` with it.

## 1. Setup and data

The development releases (dated 2022 or earlier) with this question's flag and a move; the inputs
are the export's features without the three release-timing features (V2-54), plus the company's
history `prev4_mean`, `n_prev`. Each release also gets `log_move` (the log of its move, at least
0.01%), whether it moved more than 5%, 10% and 20%, and whether its window was long.

In [ ]:
# Setup: libraries, the releases the model learns from, the inputs, MLflow
import os
import re
import tempfile
import warnings
from pathlib import Path

os.environ.setdefault("MLFLOW_DISABLE_AGENT_HINT", "1")  # keep MLflow's tips out of the outputs

import lightgbm as lgb
import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from sklearn.base import BaseEstimator, ClassifierMixin, RegressorMixin, TransformerMixin
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import brier_score_loss, log_loss
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_limits

from sentinel import config, eda
from sentinel.models import dilution  # the approved 007 code: Bands, provenance (V2-52)
from sentinel.models import results_move  # the approved model's code (V2-55)
from sentinel.models.dated import Dated

# SENTINEL_TRIAL=1 runs the code quickly (one setting, one seed): a check of the code, not results
TRIAL = os.environ.get("SENTINEL_TRIAL") == "1"
SEED, THREADS = results_move.SEED, results_move.THREADS  # LightGBM on 8 of the 16 cores
threadpool_limits(THREADS)
warnings.filterwarnings("ignore", category=RuntimeWarning)  # nan-quantiles of empty columns
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
P = config.paths()

QUESTION, ANSWER, ELIG = results_move.QUESTION, results_move.ANSWER, results_move.ELIGIBILITY
EVENTS = results_move.EVENTS  # a move above 5%, 10% and 20% beyond the market (D62)
ABOVE = {e: f"above_{round(e * 100)}" for e in EVENTS}
FLOOR = results_move.FLOOR  # the log of a move uses at least 0.01% (a handful are exactly 0)

raw = eda.load(QUESTION)
rows = results_move.learnable(raw)  # eligible releases with a move, sorted, inputs as floats

# Inputs: the export's features without the release-timing features (V2-54), plus the company's
# history (declared, FR-002). Never the answer, its dates, identifiers, flags, the sector (current
# SIC, not point-in-time) or cum_split_after (look-ahead)
TIMING, HISTORY = results_move.TIMING, results_move.HISTORY
FEATURES = results_move.features(rows)
NEVER = {"move", "close_after", "filing_date", "report_date", "cik", "as_of", "sector",
         "cum_split_after", "month_end", "release_eve", "latest", "coverage", "filer_type",
         "foreign_filer", "active_filer", "excluded", "eligible", "eligible_volatility",
         "eligible_dilution", *TIMING}
assert not NEVER & set(FEATURES) and len(FEATURES) == 94, len(FEATURES)
FAMILY = {c: eda.family(c) or "history" for c in FEATURES}
derived = {"log_move": results_move.log_move(rows),
           **{ABOVE[e]: results_move.above(rows, e) for e in EVENTS},
           "long_window": (rows["filing_date"] - rows["report_date"]).dt.days >= 5}
rows = pd.concat([rows, pd.DataFrame(derived, index=rows.index)], axis=1).copy()  # one block

print(f"{len(rows):,} releases, {rows.cik.nunique():,} companies; median move "
      f"{rows[ANSWER].median():.2%}; moved more than 5%, 10%, 20%: "
      + ", ".join(f"{rows[ABOVE[e]].mean():.1%}" for e in EVENTS))
print(f"{len(FEATURES)} inputs: {len(FEATURES) - len(HISTORY)} features (without {TIMING}, "
      f"V2-54) and {HISTORY}; {int((rows.n_prev == 0).sum()):,} first releases (no history); "
      f"{rows.long_window.mean():.1%} long windows (8-K 5+ days after the release)")
display(rows.assign(year=rows.as_of.dt.year).groupby("year").agg(
    releases=("cik", "size"), companies=("cik", "nunique"), median_move=(ANSWER, "median"),
    above_5=("above_5", "mean"), above_10=("above_10", "mean"), above_20=("above_20", "mean")))

# MLflow: runs logged to data/mlflow.db, artifacts to data/mlartifacts (research R12)
mlflow.set_tracking_uri(f"sqlite:///{P.mlflow_db}")
EXPERIMENT = "006-results-move" + ("-trial" if TRIAL else "")  # trial runs kept apart
if mlflow.get_experiment_by_name(EXPERIMENT) is None:
    mlflow.create_experiment(EXPERIMENT,
                             artifact_location=(P.mlflow_db.parent / "mlartifacts").as_uri())
mlflow.set_experiment(EXPERIMENT)
SRC = eda.EXPORT / "results_move.parquet"
TAGS = {**dilution.provenance(SRC), "level3_estimator": f"lightgbm {lgb.__version__}",
        "rules": "Dated (V2-49); one SE of the yearly gaps (V2-50); bands by p10 (V2-53); "
                 "no timing features (V2-54)",
        "trial": str(TRIAL)}
print(TAGS)

## 2. Walk-forward folds

For each test year Y the releases fall in blocks by their date and the day their move was known
(`close_after`, `eda.folds`): **inner** (move known before Y−1: the forecasts learn from these),
**gap** (dated before Y−1, known during Y−1: left out), **valid** (dated in Y−1, known before Y:
settings, calibrations and band edges), **purged** (known in Y or later: left out) and **test**
(dated in Y: scored once). Windows last 2–14 days, so the gap and the purge hold only a few
releases. Every fitted step is a `Dated` with its cutoff: 1 January of Y−1 for the forecasts,
1 January of Y for settings, calibrations and band edges.

The helpers below are shared by every level: each level gives one **forecast of the log move**;
three Platt calibrations on the validation year turn it into the three chances (research R3),
which are then put in order (the chance above 20% at most the chance above 10%, at most the
chance above 5%).

In [ ]:
# Walk-forward folds cut by dates and checked (eda.folds, eda.check_folds), and the helpers
FOLDS = eda.folds(rows, QUESTION)
display(eda.check_folds(rows, FOLDS, QUESTION))  # refuses any block out of time order
for f in FOLDS.values():
    f["train"] = f["inner"] | f["gap"] | f["valid"]  # every move known before the test year
fold_table = pd.DataFrame({year: {k: int(m.sum()) for k, m in f.items()}
                           for year, f in FOLDS.items()}).T
display(fold_table)
expected = eda.purge_table(rows, "close_after")
assert (fold_table[["train", "purged", "test"]].values
        == expected[["train_rows", "purged", "test_rows"]].values).all()
print("fold counts equal the EDA's purge table")


def valid_start(year: int) -> pd.Timestamp:
    """The cutoff of the forecasts judged on year `year`: moves known before its validation year."""
    return pd.Timestamp(year - 1, 1, 1)


def test_start(year: int) -> pd.Timestamp:
    """The cutoff of the settings, calibrations and band edges used on year `year`."""
    return pd.Timestamp(year, 1, 1)


def calibrate(score, mask, year: int) -> dict:
    """Platt scaling per event (research R3): a logistic fit of "moved more than X" on the level's
    forecast, on the validation year's releases (all known before the test year). The approved
    code, `results_move.calibrate`, for every level."""
    return results_move.calibrate(score, rows[mask], test_start(year))


def chances(cals: dict, score, mask) -> tuple[dict, int]:
    """The three chances of the releases of `mask`, put in order (`results_move.in_order`), and
    how many needed it."""
    frame = rows.loc[mask, ["as_of"]].assign(score=score)
    return results_move.in_order({e: cals[e].predict(frame) for e in EVENTS})


def scores(mask, p: dict, size=None) -> dict:
    """Per event the log loss and Brier score; their mean log loss decides (V2-41). Beside them the
    rank correlation of the chance above 10% with the move and, from the forecast size, the mean
    absolute and mean absolute log errors."""
    out = {}
    for e in EVENTS:
        yt, pe = rows.loc[mask, ABOVE[e]].to_numpy(), np.clip(p[e], 1e-6, 1 - 1e-6)
        out[f"log_loss_{round(e * 100)}"] = log_loss(yt, pe, labels=[0, 1])
        out[f"brier_{round(e * 100)}"] = brier_score_loss(yt, pe)
    out["log_loss_mean"] = float(np.mean([out[f"log_loss_{round(e * 100)}"] for e in EVENTS]))
    move = rows.loc[mask, ANSWER].to_numpy()
    out["rank_corr"] = float(pd.Series(p[0.10]).corr(pd.Series(move), method="spearman"))
    if size is not None:
        out["mae"] = float(np.mean(np.abs(size - move)))
        out["mae_log"] = float(np.mean(np.abs(np.log(np.maximum(size, FLOOR))
                                              - np.log(np.maximum(move, FLOOR)))))
    return out


def valid_loss(score, mask) -> float:
    """How settings are chosen (research R5): the validation year's mean log loss of the three
    chances after a Platt scaling on that same year (`results_move.valid_loss`)."""
    return results_move.valid_loss(score, rows[mask])


RESULTS, FITTED, TEST_P, TEST_SCORE, PUT_IN_ORDER = [], {}, {}, {}, {}


def record(level: str, year: int, p: dict, size, params: dict, extra=None) -> dict:
    """Keep a level's test-year chances and scores, and log them as one MLflow run."""
    m = scores(FOLDS[year]["test"], p, size)
    RESULTS.append({"level": level, "year": year, **m})
    TEST_P[(level, year)] = p
    with mlflow.start_run(run_name=f"{level}-{year}"):
        mlflow.set_tags({**TAGS, "level": level, "test_year": year})
        mlflow.log_params(params)
        mlflow.log_metrics({**m, **(extra or {})})
    return m


def evaluate(level: str, year: int, fit: dict) -> dict:
    """Score a fitted level on its test year: the forecast, the three chances, the typical size."""
    f = FOLDS[year]
    score = fit["model"].predict(rows[f["test"]])
    p, fixed = chances(fit["cals"], score, f["test"])
    FITTED[(level, year)], TEST_SCORE[(level, year)] = fit, score
    PUT_IN_ORDER[(level, year)] = fixed
    return record(level, year, p, np.exp(score), fit["params"], {"put_in_order": fixed})


def fit_level(make, features, f, year, grid, start=None) -> dict:
    """Learn from the inner releases (moves known before the validation year), choose the setting
    by predicting the validation year, calibrate there (research R5). With `start`, only inner
    releases dated from it (the 2012 check)."""
    inner = f["inner"] if start is None else f["inner"] & (rows["as_of"] >= start)
    tried = {}
    for params in grid:
        model = Dated(make(**params), QUESTION, features)
        model.fit(rows[inner], valid_start(year), y=rows.loc[inner, "log_move"])
        s = model.predict(rows[f["valid"]])
        tried[str(params)] = (valid_loss(s, f["valid"]), params, model, s)
    _, params, model, s_valid = min(tried.values(), key=lambda t: t[0])
    return {"model": model, "cals": calibrate(s_valid, f["valid"], year), "params": params,
            "valid_score": s_valid, "features": features,
            "tried": {k: round(v[0], 5) for k, v in tried.items()}}

## 3. Level 0: the naive baseline (and the base rates beside it)

**The base rates** give every release the same three chances: the shares of the releases known
before the test year that moved more than 5%, 10% and 20%. They are not a level, but the floor
every level must beat (spec SC-004).

**The baseline** (pre-registered): the log of `prev4_mean`, the mean of the company's previous
four moves; a company's first release gets the typical move of the releases it learned from. Its
three chances come from the calibrations on the validation year, like every level's.

In [ ]:
# Level 0: the pre-registered naive baseline, and the constant base rates beside it (the floor)
class Rate(BaseEstimator, ClassifierMixin):
    """The share of the releases it learns from that moved more than an event's size: the same
    chance for every release."""

    def fit(self, X, y):
        self.rate_ = float(np.mean(y))
        return self

    def predict_proba(self, X):
        p = np.full(len(X), self.rate_)
        return np.column_stack([1 - p, p])


class PreviousFour(BaseEstimator, RegressorMixin):
    """The log of the mean of the company's previous four moves; a company's first release gets
    the typical (median) log move of the releases it learned from."""

    def fit(self, X, y):
        self.first_ = float(np.median(y))
        return self

    def predict(self, X):
        prev4 = np.asarray(X, dtype=float)[:, 0]
        return np.where(np.isfinite(prev4), np.log(np.maximum(prev4, FLOOR)), self.first_)


for year, f in FOLDS.items():
    train, test = rows[f["train"]], rows[f["test"]]
    p = {e: Dated(Rate(), QUESTION, []).fit(train, test_start(year), y=train[ABOVE[e]])
         .predict(test) for e in EVENTS}
    record("base_rate", year, p, np.full(len(test), train[ANSWER].median()),
           {f"rate_{round(e * 100)}": round(float(p[e][0]), 4) for e in EVENTS})
    evaluate("baseline", year, fit_level(PreviousFour, ["prev4_mean"], f, year, [{}]))
display(pd.DataFrame(RESULTS).set_index(["level", "year"]))

## 4. Level 1: a rule

The EDA's advice in one line: **the company's own typical results-day move, trusted more the more
releases it has, combined with how much the stock moves on ordinary days.** The history is the log
of `prev4_mean` pulled toward the typical release's log move by the weight `m / (m + k)`, with
`m` the number of earlier moves (up to four): with `k = 2`, a company with one earlier move gets a
third of its own history, one with four gets two thirds. Then

log move ≈ a + b · history + c · log(`vol_63`)

with `a`, `b`, `c` fitted by least squares on the inner releases and `k` chosen on the validation
year. Two weights say how much history and volatility each count.

In [ ]:
# Level 1: the rule (research R5): the company's own typical move, trusted more the more releases
# it has, combined with how much the stock moves on ordinary days
class Rule(BaseEstimator, RegressorMixin):
    """log move = a + b * history + c * log(vol_63); history is log(prev4_mean) pulled toward the
    typical release's log move by m / (m + k), m = min(n_prev, 4); a, b, c by least squares."""

    def __init__(self, k=2):
        self.k = k

    def _parts(self, X):
        X = np.asarray(X, dtype=float)
        prev4, n_prev, vol = X[:, 0], X[:, 1], X[:, 2]
        m = np.minimum(np.nan_to_num(n_prev), 4)
        w = m / (m + self.k)
        own = np.where(np.isfinite(prev4), np.log(np.maximum(prev4, FLOOR)), self.typical_)
        vol = np.where(np.isfinite(vol) & (vol > 0), vol, self.vol_)
        return np.column_stack([w * own + (1 - w) * self.typical_, np.log(vol)])

    def fit(self, X, y):
        self.typical_ = float(np.median(y))
        v = np.asarray(X, dtype=float)[:, 2]
        self.vol_ = float(np.nanmedian(v[v > 0]))
        self.lr_ = LinearRegression().fit(self._parts(X), y)
        return self

    def predict(self, X):
        return self.lr_.predict(self._parts(X))


RULE = ["prev4_mean", "n_prev", "vol_63"]
KS = [2] if TRIAL else [1, 2, 4]
weights = {}
for year, f in FOLDS.items():
    fit = fit_level(Rule, RULE, f, year, [{"k": k} for k in KS])
    evaluate("rule", year, fit)
    lr = fit["model"].estimator.lr_
    weights[year] = {"k": fit["params"]["k"], "a": lr.intercept_, "b (history)": lr.coef_[0],
                     "c (volatility)": lr.coef_[1], "validation loss by k": fit["tried"]}
display(pd.DataFrame(weights).T)
display(pd.DataFrame(RESULTS).query("level == 'rule'").set_index("year"))

## 5. Level 2: a classic model

A ridge regression of the log move on all the inputs, with the model-side cleaning of V2-33,
fitted on the inner releases only: the positive measures of a move's size (`prev4_mean` and the
volatilities) as logs (the log move rises with their logs); one of each pair of near-duplicate
columns (|Spearman| ≥ 0.9 on the 2019 fold's inner releases, fixed for every fold); every column
cut at its 1st and 99th percentiles; missing values filled with the median plus a "was missing"
indicator; standardised. The strength of the ridge penalty is chosen on the validation year.

In [ ]:
# Level 2: a ridge regression of the log move with the model-side cleaning of V2-33
VOL = ["vol_21", "vol_63", "vol_252", "range_vol_21", "idio_vol_252"]
LOGGED = ["prev4_mean", *VOL]  # positive measures of a move's size: the log move rises with their logs


class Logs(BaseEstimator, TransformerMixin):
    """The log (at least FLOOR) of the columns `cols` among `columns`; the others unchanged."""

    def __init__(self, columns, cols):
        self.columns, self.cols = columns, cols

    def fit(self, X, y=None):
        self.idx_ = [self.columns.index(c) for c in self.cols if c in self.columns]
        return self

    def transform(self, X):
        X = np.array(X, dtype=float)
        X[:, self.idx_] = np.log(np.maximum(X[:, self.idx_], FLOOR))
        return X


class Clip(BaseEstimator, TransformerMixin):
    """Cut each column at its 1st and 99th percentiles of the rows it was fitted on."""

    def __init__(self, lower=0.01, upper=0.99):
        self.lower, self.upper = lower, upper

    def fit(self, X, y=None):
        X = np.asarray(X, dtype=float)
        self.lo_ = np.nanquantile(X, self.lower, axis=0)
        self.hi_ = np.nanquantile(X, self.upper, axis=0)
        return self

    def transform(self, X):
        return np.clip(np.asarray(X, dtype=float), self.lo_, self.hi_)


# One of each |Spearman| >= 0.9 pair, found on the 2019 fold's inner releases (features against
# features, no answer used; fixed for every fold), keeping the log versions and key features
KEEP = {"log_addv20", "log_market_cap", "log_close", "idio_vol_252", "vol_63", "prev4_mean",
        "n_prev"}
pairs = eda.redundant_pairs(rows[FOLDS[2019]["inner"]], FEATURES, threshold=0.9, seed=SEED)
redundant = []
for a, b in zip(pairs.a, pairs.b, strict=True):
    if a in redundant or b in redundant or (a in KEEP and b in KEEP):
        continue
    redundant.append(a if b in KEEP else b)
LINEAR = [c for c in FEATURES if c not in redundant]
print(f"{len(LINEAR)} columns; left out as redundant: {redundant}")


def ridge(alpha: float, columns=None):
    """The classic model on `columns` (default: LINEAR)."""
    return make_pipeline(Logs(list(columns or LINEAR), LOGGED), Clip(),
                         SimpleImputer(strategy="median", add_indicator=True), StandardScaler(),
                         Ridge(alpha=alpha))


ALPHAS = [100.0] if TRIAL else [1.0, 10.0, 100.0, 1000.0]
for year, f in FOLDS.items():
    fit = fit_level(ridge, LINEAR, f, year, [{"alpha": a} for a in ALPHAS])
    evaluate("ridge", year, fit)
    print(year, "validation loss by alpha:", fit["tried"], "-> alpha", fit["params"]["alpha"])
display(pd.DataFrame(RESULTS).query("level == 'ridge'").set_index("year"))

## 6. Level 3: LightGBM

Gradient-boosted trees (D60) forecasting the log move from all the inputs, with fixed seeds;
missing values are left to the trees. They learn from the inner releases; the settings (leaves,
releases per leaf) and the number of trees are chosen by predicting the validation year, never by
early stopping inside the fit (V2-49). The design check on 2016–2018 chose 200 trees, the fewest it
tried, so the list now starts at 50. This is the approved model's code,
`src/sentinel/models/results_move.py` (V2-55).

In [ ]:
# Level 3: LightGBM's regression of the log move (D60), the approved model's code
# (src/sentinel/models/results_move.py, V2-55): it learns from the inner releases, chooses its
# setting and number of trees by predicting the validation year (no early stopping, V2-49), and is
# calibrated there
if TRIAL:
    results_move.GRID, results_move.TREES = results_move.GRID[:1], [100, 200]
print("settings tried:", results_move.GRID, "| trees:", results_move.TREES)


def as_fit(m: results_move.ResultsMoveModel) -> dict:
    """The packaged model in the shape the notebook's helpers use."""
    return {"model": m.forecast, "cals": m.calibrators, "setting": m.setting,
            "params": {**m.setting["params"], "trees": m.setting["trees"]},
            "valid_score": m.valid_score, "features": m.features, "tried": m.tried,
            "packaged": m}


for year, f in FOLDS.items():
    fit = as_fit(results_move.fit(rows, year))
    evaluate("lightgbm", year, fit)
    best = sorted(fit["tried"].items(), key=lambda kv: kv[1])[:3]
    print(year, "chosen:", fit["setting"], "| best three on the validation year:", best)
display(pd.DataFrame(RESULTS).query("level == 'lightgbm'").set_index("year"))


## 7. Comparison and choice

Each level's measures per test year, the base rates on top. The deciding measure is the mean log
loss of the three chances (V2-41). The choice (V2-50): of the levels whose gap to the best is
within one standard error of their four yearly gaps, the simplest. A more complex level wins when
it beats a simpler one steadily, year by year, whatever the size of its lead.

In [ ]:
# Comparison of the four levels and the year-by-year one-standard-error choice (V2-50)
ORDER = ["baseline", "rule", "ridge", "lightgbm"]  # simplest first
res = pd.DataFrame(RESULTS)
MEASURES = ["log_loss_mean", "log_loss_5", "log_loss_10", "log_loss_20", "brier_5", "brier_10",
            "brier_20", "rank_corr", "mae", "mae_log"]
for measure in MEASURES:
    print(measure + ("  (deciding, V2-41)" if measure == "log_loss_mean" else ""))
    display(res.pivot(index="level", columns="year", values=measure).loc[["base_rate", *ORDER]])

LL = res.pivot(index="year", columns="level", values="log_loss_mean")
best = LL[ORDER].mean().idxmin()
gap = LL[ORDER].sub(LL[best], axis=0)  # how much worse than the best, per test year
summary = pd.DataFrame({
    "mean log loss": LL[ORDER].mean(),
    "gap to the best (mean)": gap.mean(),
    "SE of the yearly gaps": gap.std() / np.sqrt(len(LL)),
    "years the best is better": (gap > 0).sum(),
}).loc[ORDER]
summary["as good as the best"] = (summary["gap to the best (mean)"]
                                  <= summary["SE of the yearly gaps"])
CHOSEN = next(level for level in ORDER if summary.loc[level, "as good as the best"])
display(summary)
print("gap to the best per test year:")
display(gap.T.loc[ORDER])
print(f"best mean log loss: {best}; CHOSEN LEVEL: {CHOSEN} (the simplest level whose gap to the "
      "best is within one standard error of its yearly gaps, V2-50)")
spread = LL[best].std() / np.sqrt(len(LL))
textbook = next(level for level in ORDER if LL[level].mean() <= LL[best].mean() + spread)
print(f"for reference, the textbook version (one standard error of the best level's own yearly "
      f"scores, {spread:.4f}) would choose: {textbook}")

CHECKS = pd.DataFrame({"beats the base rates": LL[CHOSEN] < LL["base_rate"],
                       "beats the baseline": LL[CHOSEN] < LL["baseline"],
                       "beats the rule": LL[CHOSEN] < LL["rule"]})
display(CHECKS)
if CHOSEN not in ("baseline", "rule") and not CHECKS["beats the rule"].all():
    print("The chosen level does not beat the rule in every test year: the findings say so and "
          "recommend the rule (spec, US1 scenario 4).")

## 8. Calibration and bands

**Bands** (V2-53): five groups of releases by the chance of a move above 10%, with edges at the
quintiles of the validation year (a `Dated` with the test year's cutoff). Calibration keeps the
forecasts' order (positive slopes), so the edges are set on the level's own forecast: the same
groups as by the chance above 10%, without re-scoring releases the calibration learned from. For
each band and test year: the releases, the forecast and the real share above 5%, 10% and 20% (its
hit rates, D47) and the median move.

**Calibration charts:** releases in ten groups by their chance; an honest chance lies on the
diagonal (forecast 30%, real share 30%).

In [ ]:
# Five bands by the chance above 10% (V2-53) with edges from the validation year, and calibration
band_rows, rising = [], {}
for year, f in FOLDS.items():
    fit = FITTED[(CHOSEN, year)]
    valid = rows.loc[f["valid"], ["as_of", "close_after"]].assign(score=fit["valid_score"])
    banding = Dated(dilution.Bands(), QUESTION, ["score"]).fit(valid, test_start(year),
                                                                y=rows.loc[f["valid"], ANSWER])
    test = rows.loc[f["test"], ["as_of", ANSWER, *ABOVE.values()]]
    band = banding.predict(test[["as_of"]].assign(score=TEST_SCORE[(CHOSEN, year)]))
    p = TEST_P[(CHOSEN, year)]
    t = (test.assign(band=band, p5=p[0.05], p10=p[0.10], p20=p[0.20]).groupby("band")
         .agg(releases=(ANSWER, "size"), forecast_5=("p5", "mean"), real_5=("above_5", "mean"),
              forecast_10=("p10", "mean"), real_10=("above_10", "mean"),
              forecast_20=("p20", "mean"), real_20=("above_20", "mean"),
              median_move=(ANSWER, "median")))
    rising[year] = {"real share above 10% rises": bool(t["real_10"].is_monotonic_increasing),
                    "median move rises": bool(t["median_move"].is_monotonic_increasing)}
    band_rows.append(t.assign(year=year).reset_index())
BAND_TABLE = pd.concat(band_rows).set_index(["year", "band"])
print(f"bands of {CHOSEN} per test year (forecast and real shares above 5%, 10%, 20%):")
display(BAND_TABLE)
display(pd.DataFrame(rising).T)

in_order = all((TEST_P[(CHOSEN, y)][0.20] <= TEST_P[(CHOSEN, y)][0.10]).all()
               and (TEST_P[(CHOSEN, y)][0.10] <= TEST_P[(CHOSEN, y)][0.05]).all() for y in FOLDS)
print("releases whose chances were put in order, per test year:",
      {y: PUT_IN_ORDER[(CHOSEN, y)] for y in FOLDS}, "| every release in order:", in_order)
assert in_order

fig, axes = plt.subplots(len(EVENTS), len(FOLDS), figsize=(16, 11))
for i, e in enumerate(EVENTS):
    for j, (year, f) in enumerate(FOLDS.items()):
        d = pd.DataFrame({"p": TEST_P[(CHOSEN, year)][e],
                          "y": rows.loc[f["test"], ABOVE[e]].to_numpy()})
        d["group"] = pd.qcut(d["p"].rank(method="first"), 10, labels=False)
        g = d.groupby("group").mean()
        top = float(max(g["p"].max(), g["y"].max()))
        ax = axes[i, j]
        ax.plot([0, top], [0, top], color="grey", lw=1, ls="--")
        ax.plot(g["p"], g["y"], marker="o")
        ax.set_title(f"above {e:.0%}, {year}")
        if j == 0:
            ax.set_ylabel("real share")
        if i == len(EVENTS) - 1:
            ax.set_xlabel("forecast chance")
fig.suptitle(f"Calibration of {CHOSEN}: ten groups of releases per test year (on the line = "
             "honest)")
plt.tight_layout()
plt.show()

## 9. What the model uses, and the checks

The strongest inputs of the ridge regression and of LightGBM (2021 fold), and a leak check: no
input outside the feature list, and no single feature near-perfect on its own for any event.

Then the two checks asked by D60 and the EDA, on the chosen level; they report, they choose
nothing (research R8): **training from 2012** instead of 2010 (the start was fixed at 2010 on
2016–2018), and the test years **with and without the long windows** (8-K filed 5 or more days
after the release).

In [ ]:
# What the models use, a leak check (research R10), and the checks of D60 and the EDA (R8)
pipe = FITTED[("ridge", 2021)]["model"].estimator
coef, imputer = pipe[-1].coef_, pipe[2]
labels = LINEAR + [f"{LINEAR[i]} (was missing)" for i in imputer.indicator_.features_]
top_coef = (pd.Series(coef, index=labels).sort_values(key=np.abs, ascending=False).head(20)
            .rename("standardised coefficient (ridge, 2021 fold)"))
display(top_coef.to_frame())

boost = FITTED[("lightgbm", 2021)]["model"].estimator
gain = pd.Series(boost.model_.booster_.feature_importance("gain", iteration=boost.trees_),
                 index=FEATURES)
top_gain = (gain / gain.sum()).sort_values(ascending=False).head(20).rename(
    "share of LightGBM gain (2021 fold)")
display(top_gain.to_frame().assign(family=lambda t: [FAMILY[c] for c in t.index]))

# Leak check: inputs are features only, and no single feature is near-perfect for any event
assert set(LINEAR) <= set(FEATURES) and not NEVER & set(FEATURES)
tr = FOLDS[2019]["train"]
single = pd.DataFrame({e: {c: eda.auc(rows.loc[tr, ABOVE[e]], rows.loc[tr, c]) for c in FEATURES
                           if rows.loc[tr, c].notna().sum() > 1000} for e in EVENTS})
suspicious = single[((single >= 0.95) | (single <= 0.05)).any(axis=1)]
print("features with a single AUC >= 0.95 or <= 0.05 on the 2019 fold's training releases:",
      list(suspicious.index))
assert suspicious.empty
display(single.sub(0.5).abs().max(axis=1).sort_values(ascending=False).head(10)
        .rename("strongest single |AUC - 0.5| of any event").to_frame())

CHECK_LEVEL = CHOSEN if CHOSEN in ("ridge", "lightgbm") else "lightgbm"


def refit(level: str, f, year: int, features=None, seed=SEED, start=None) -> dict:
    """The level refitted with the fold's chosen setting, on other features, seed or start."""
    fitted = FITTED[(level, year)]
    if level == "lightgbm":
        return as_fit(results_move.fit(rows, year, features, seed, fitted["setting"],
                                       start=start))
    if level == "ridge":
        cols = list(features or LINEAR)
        return fit_level(lambda alpha: ridge(alpha, cols), cols, f, year, [fitted["params"]],
                         start)
    if level == "rule":
        return fit_level(Rule, RULE, f, year, [fitted["params"]], start)
    return fit_level(PreviousFour, ["prev4_mean"], f, year, [{}], start)


def test_loss(fit: dict, year: int, mask=None) -> float:
    """The test year's mean log loss of a fit's three chances (within `mask` if given)."""
    test = FOLDS[year]["test"] if mask is None else FOLDS[year]["test"] & mask
    p, _ = chances(fit["cals"], fit["model"].predict(rows[test]), test)
    return scores(test, p)["log_loss_mean"]


start12 = pd.Timestamp(2012, 1, 1)
CHECK_2012 = pd.DataFrame({year: {"from 2010 (as fitted)": test_loss(FITTED[(CHOSEN, year)], year),
                                  "from 2012": test_loss(refit(CHOSEN, f, year, start=start12),
                                                         year)}
                           for year, f in FOLDS.items()}).T
CHECK_2012["2012 minus 2010"] = CHECK_2012["from 2012"] - CHECK_2012["from 2010 (as fitted)"]
print(f"{CHOSEN}: mean log loss on the test years, learning from 2010 or from 2012")
display(CHECK_2012)

short = ~rows["long_window"]
WINDOWS = pd.DataFrame({year: {
    "all releases": test_loss(FITTED[(CHOSEN, year)], year),
    "without long windows": test_loss(FITTED[(CHOSEN, year)], year, short),
    "long-window releases": int((f["test"] & rows["long_window"]).sum())}
    for year, f in FOLDS.items()}).T.astype({"long-window releases": int})
print(f"{CHOSEN}: mean log loss on the test years with and without the long windows")
display(WINDOWS)
with mlflow.start_run(run_name="checks"):
    mlflow.set_tags({**TAGS, "level": CHOSEN, "check": "2012 start; long windows"})
    mlflow.log_metrics({f"from2012_minus_2010_{y}": v
                        for y, v in CHECK_2012["2012 minus 2010"].items()})

## 9b. The hypotheses (V2-46, docs/hypotheses.md)

Each hypothesis names a family of inputs and why it should matter for the size of a results-day
move. **Family check:** the chosen level (LightGBM if a rule level was chosen) is refitted without
the family on every fold, with that fold's settings, five seeds; the family **adds** if the mean
log loss is worse without it, averaged over the four test years, by more than one standard error
of the four yearly differences. **Direction check:** the rank correlation of the key feature with
the move in each test year; the expected sign in at least three of the four. **Verdict:**
supported (adds, right direction), not supported (adds nothing), opposite (the direction is
reversed in at least three years). H-R6 expects no effect: supported when its families add
nothing. H-R1 also needs the baseline to rank the moves (rank correlation above 0).

| # | Family | Why | Expected |
|---|---|---|---|
| H-R1 | history (`prev4_mean`, `n_prev`) | some companies habitually surprise the market | larger past moves, larger next move |
| H-R2 | volatility (`vol_21`, `vol_63`, `vol_252`, `range_vol_21`, `idio_vol_252`) | a stock that is volatile on ordinary days is also volatile on its results day | higher idiosyncratic volatility, larger move |
| H-R3 | size and trading | small companies are followed by fewer analysts, so their results surprise more | smaller, larger move |
| H-R4 | financials | loss-making and cash-poor companies' results are more uncertain | loss-making, larger move |
| H-R5 | macro | in fearful markets investors react more strongly to any news | higher VIX, larger move |
| H-R6 | insider, short volume | no clear mechanism | no effect |

In [ ]:
# The hypotheses for the results move (docs/hypotheses.md): family checks and direction checks
BASE = FEATURES if CHECK_LEVEL == "lightgbm" else LINEAR
SEEDS = range(5) if CHECK_LEVEL == "lightgbm" and not TRIAL else range(1)  # ridge has no seed
GROUP = {
    "H-R1 history": HISTORY,
    "H-R2 volatility": VOL,
    "H-R3 size and trading": [c for c in FEATURES if FAMILY[c] in ("size", "trading")],
    "H-R4 financials": [c for c in FEATURES if FAMILY[c] == "financials"],
    "H-R5 macro": [c for c in FEATURES if FAMILY[c] == "macro"],
    "H-R6 insider and short volume": [c for c in FEATURES
                                      if FAMILY[c] in ("insider", "short volume")],
}
VARIANTS = {"all": BASE, **{h: [c for c in BASE if c not in g] for h, g in GROUP.items()}}
CHECK_LL = {(name, year): np.mean([test_loss(refit(CHECK_LEVEL, FOLDS[year], year, cols, s), year)
                                   for s in SEEDS])
            for name, cols in VARIANTS.items() for year in FOLDS}
family_rows = []
for h, g in GROUP.items():
    d = pd.Series({year: CHECK_LL[(h, year)] - CHECK_LL[("all", year)] for year in FOLDS})
    se = d.std() / np.sqrt(len(d))
    family_rows.append({"check": h, "inputs": len([c for c in g if c in BASE]),
                        **{str(year): v for year, v in d.items()}, "mean": d.mean(), "se": se,
                        "adds": bool(d.mean() > se)})
FAMILY_TABLE = pd.DataFrame(family_rows).set_index("check")
print(f"mean log loss without the family minus with it ({CHECK_LEVEL}, seeds {list(SEEDS)}): "
      "positive means the family helps")
display(FAMILY_TABLE)

# Direction checks: rank correlation of each key feature with the move, per test year
KEY = {"H-R1": ("prev4_mean", +1), "H-R2": ("idio_vol_252", +1),
       "H-R3": ("log_market_cap", -1), "H-R4": ("loss_making", +1), "H-R5": ("vix", +1)}
test_rows = rows[np.logical_or.reduce([f["test"].to_numpy() for f in FOLDS.values()])]
rho = eda.rank_by_year(test_rows, [k for k, _ in KEY.values()], ANSWER)
display(rho)

baseline_ranks = bool((res.query("level == 'baseline'")["rank_corr"] > 0).all())
verdicts = []
for h, (feature, sign) in KEY.items():
    check = next(c for c in FAMILY_TABLE.index if c.startswith(h))
    right = int((np.sign(rho.loc[feature]) == sign).sum())
    wrong = int((np.sign(rho.loc[feature]) == -sign).sum())
    adds = bool(FAMILY_TABLE.loc[check, "adds"]) and (h != "H-R1" or baseline_ranks)
    verdict = "opposite" if wrong >= 3 else ("supported" if adds and right >= 3
                                             else "not supported")
    verdicts.append({"hypothesis": check, "expected": f"{feature} {'+' if sign > 0 else '−'}",
                     "change without (mean ± SE)": f"{FAMILY_TABLE.loc[check, 'mean']:+.4f} ± "
                                                   f"{FAMILY_TABLE.loc[check, 'se']:.4f}",
                     "expected sign in years": f"{right} of {len(FOLDS)}", "verdict": verdict})
h6 = FAMILY_TABLE.loc["H-R6 insider and short volume"]
verdicts.append({"hypothesis": "H-R6 insider and short volume", "expected": "no effect",
                 "change without (mean ± SE)": f"{h6['mean']:+.4f} ± {h6['se']:.4f}",
                 "expected sign in years": "–",
                 "verdict": "not supported" if h6["adds"] else "supported"})
VERDICTS = pd.DataFrame(verdicts).set_index("hypothesis")
print("H-R1 also needs the baseline to rank the moves (rank correlation above 0) in every test "
      "year:", baseline_ranks)
display(VERDICTS)
with mlflow.start_run(run_name="hypotheses"):
    mlflow.set_tags({**TAGS, "level": CHECK_LEVEL, "check": "hypotheses"})
    for check, r in FAMILY_TABLE.iterrows():
        mlflow.log_metric(re.sub(r"[^A-Za-z0-9_]", "_", check)[:60] + "_mean", r["mean"])

## 10. Frozen model and dry run

The chosen level made for the holdout exactly as for a test year (target year 2023): the forecast
learns from moves known before 2022; its settings, calibrations and band edges come from the
releases of 2022 (known before 2023). It is one object of `Dated` steps with one `predict`
function (contract: `specs/006-results-move-model/contracts/model-contract.md`), fitted twice to
show the two fits agree. The dry run scores rows without answers: the 2023+ release eves of
`feature_snapshot.parquet` and the newest row of each company still trading (`latest.parquet`).
Their earlier moves after 2022 are sealed until the holdout test, so `prev4_mean` and `n_prev` are
given as missing; the holdout command (010) computes them. This is the approved code in
`src/sentinel/models/results_move.py` (V2-55); `sentinel train results_move` makes the same model.

In [ ]:
# The frozen model: the approved code (src/sentinel/models/results_move.py, V2-55) made for the
# holdout year exactly as for a test year: one object of Dated steps and one predict function
assert CHOSEN == "lightgbm", CHOSEN  # the level the researcher approved (V2-55)
made_from = {k: TAGS[k] for k in ("git_commit", "export_date", "data_hash")}
frozen = results_move.fit(rows, results_move.TARGET_YEAR, provenance=made_from)
again = results_move.fit(rows, results_move.TARGET_YEAR, provenance=made_from)
print(f"level {frozen.level}; cutoffs: forecast {frozen.forecast.cutoff.date()}, calibrations "
      f"{frozen.calibrators[0.10].cutoff.date()}, bands {frozen.bands.cutoff.date()}; the "
      f"forecast learned from moves known through {frozen.trained_through}; "
      f"setting {frozen.setting}")

# Dry run on rows without answers (research R11)
snap = pq.read_table(eda.EXPORT / "feature_snapshot.parquet",
                     filters=[("as_of", ">", eda.DEV_END), ("release_eve", "=", True)]).to_pandas()
latest = pd.read_parquet(eda.EXPORT / "latest.parquet")
latest["as_of"] = pd.to_datetime(latest["as_of"])
latest = latest[latest["as_of"] >= latest["as_of"].max() - pd.Timedelta(days=7)]  # still trading
CHANCE_COLS = results_move.CHANCES
for name, frame in {"2023+ release eves": snap, "latest (still trading)": latest}.items():
    assert ANSWER not in frame.columns
    frame = frame.assign(prev4_mean=np.nan, n_prev=np.nan)  # sealed until the holdout test
    out, out2 = results_move.predict(frozen, frame), results_move.predict(again, frame)
    scored = out[out["eligible"]]
    eligible = int(frame[ELIG].fillna(False).astype(bool).sum())
    assert len(scored) == eligible and scored[CHANCE_COLS].notna().all().all()
    assert scored[CHANCE_COLS].stack().between(0, 1).all()
    assert ((scored["p_move_20"] <= scored["p_move_10"])
            & (scored["p_move_10"] <= scored["p_move_5"])).all()
    assert scored["move_band"].between(1, 5).all()
    assert out[CHANCE_COLS].equals(out2[CHANCE_COLS])  # two fits, identical forecasts
    bands = dict(sorted((int(b), int(n)) for b, n in scored["move_band"].value_counts().items()))
    print(f"{name}: {len(frame):,} rows, {eligible:,} eligible, all scored, chances in [0, 1] and "
          f"in order, two fits identical; mean chances "
          f"{scored[CHANCE_COLS].mean().round(3).tolist()}; bands {bands}")

# The guard at work: the frozen model refuses a release dated before its cutoff
try:
    results_move.predict(frozen, rows[rows.as_of.dt.year == 2021].head(1))
    raise RuntimeError("a 2021 release was scored")
except AssertionError as err:
    print("refused as expected:", str(err)[:90])

with tempfile.TemporaryDirectory() as tmp, mlflow.start_run(run_name=f"frozen-{CHOSEN}"):
    path = Path(tmp) / "results_move_model.joblib"
    results_move.save(frozen, path)
    mlflow.set_tags({**TAGS, "level": CHOSEN, "frozen": "approved (V2-55)"})
    mlflow.log_params({"level": CHOSEN, "trained_through": frozen.trained_through,
                       "n_features": len(frozen.features),
                       "forecast_cutoff": str(frozen.forecast.cutoff.date())})
    mlflow.log_artifact(str(path))
print("frozen model saved in MLflow (approved, V2-55); `sentinel train results_move` makes the "
      "same model")


### Findings

*Export of 9 November 2026 (data hash `60d44c447ca6`). First full run at commit `8fa4a8c` (424 s);
LightGBM approved by the researcher the same day (V2-55); re-run at commit `ad232a7`, which imports
the approved code from `src/sentinel/models/results_move.py`, with every number identical (475 s,
under the memory cap). Every number is an aggregate from the section named in brackets.*

#### In one paragraph

**LightGBM gives the best chances in every test year and is the chosen level** (V2-50). Its mean
log loss of the three chances is 0.404, against 0.409 for the ridge regression, 0.421 for the
rule, 0.433 for the pre-registered baseline and 0.469 for the base rates. It beats the ridge, the
next best, in each of the four years (by 0.003 to 0.009). Its chances are honest in the middle and a
little too high at the top: releases in band 5 moved more than 10% in 39–41% of cases, against a
forecast of 41–45%. Five of the six hypotheses find what they expected, and the sixth (insider
and short volume: no effect) holds as well. The model is frozen for the holdout year and passes
the dry run.

#### The levels [3–7]: mean log loss of the three chances (deciding, V2-41; lower is better)

| Level | 2019 | 2020 | 2021 | 2022 | Mean |
|---|---|---|---|---|---|
| Base rates (the floor) | 0.4553 | 0.4764 | 0.4333 | 0.5095 | 0.4686 |
| 0 Baseline (`prev4_mean`) | 0.4058 | 0.4442 | 0.4118 | 0.4715 | 0.4333 |
| 1 Rule (history + volatility) | 0.3983 | 0.4361 | 0.3959 | 0.4529 | 0.4208 |
| 2 Ridge | 0.3919 | 0.4229 | 0.3883 | 0.4341 | 0.4093 |
| **3 LightGBM** | **0.3832** | **0.4192** | **0.3819** | **0.4308** | **0.4038** |

- Every level beats the one below it in every test year; 2020 (the pandemic) and 2022 (falling
  markets) are hard for all of them.
- The same order holds for the 5% and 10% events in every year, by log loss and by Brier score.
  For the rare 20% event the levels are closer: in 2020 the rule is worse than the baseline, and
  LightGBM's Brier score ties the ridge's in 2020 and is 0.0002 worse in 2022 (its log loss is
  better in every year).
- **Ranking**: the rank correlation of the chance above 10% with the move is 0.38–0.46 for
  LightGBM, against 0.29–0.39 for the baseline.
- **Size**: the typical size LightGBM forecasts is off by 3.9–4.7 points on average (mean absolute
  error 0.039–0.047), better than one constant for everyone (0.042–0.052) in every year; the
  baseline was not (the EDA's surprise, confirmed: 0.046–0.052).
- The rule is worth having: its weights say a company's own history counts most (b 0.54–0.90:
  k = 4 in three folds, 1 in 2022) and ordinary-day volatility adds (c 0.39–0.42 in every fold).

#### The choice [7]

- **Year-by-year rule (V2-50, decided): LightGBM.** Its lead over the ridge is 0.0055 on average
  with a standard error of 0.0012 over the four yearly leads: it wins steadily, every year.
- **For reference, the textbook rule would keep the ridge**: one standard error of LightGBM's own
  yearly scores is 0.0125, larger than the whole gap to the ridge, because the hard years (2020,
  2022) move every level together. This is the case V2-50 was decided for; the ridge is the
  simpler alternative, 1.4% worse in log loss.
- The chosen level beats the base rates, the baseline and the rule in every test year (SC-004).

#### Bands and calibration [8]

Five bands by the chance above 10% (V2-53), edges from the year before each test year:

| Band | Real share above 10% | Real share above 20% | Median move |
|---|---|---|---|
| 1 · Very low | 0.9–2.3% | 0.0–0.4% | 1.7–2.1% |
| 5 · Very high | 38.7–40.9% | 13.3–14.2% | 7.5–8.2% |

- In every test year the real share above 10% and the median move rise from band 1 to band 5
  (SC-005), and every release's chances are in order with no correction needed (SC-006).
- **Calibration**: close to the diagonal for most releases; band 5 forecasts are 2–5 points too
  high for the 10% event (41–45% forecast, 39–41% real), and in 2021, after the turbulent 2020,
  band 1's chance above 5% was too high (20% forecast, 15% real). The calibration learns from one
  year, so a calm year after a wild one (or the reverse) shifts it. Reported, not adjusted: a fix
  chosen by looking at the test years would flatter them.
- **Band sizes move with the market** because the edges come from the year before: in 2021 a
  calmer year put 29% of releases in band 1; in 2022 a wilder one put 39% in band 5.

#### What the model uses [9]

- The company's own history (`prev4_mean`) carries 43% of LightGBM's gain; ordinary-day
  volatility (`range_vol_21`, `vol_252`, `vol_21`, `idio_vol_252`, `vol_63`) 21%; then distance
  from the 12-month high, the current ratio, traded value, receivables, beta. The ridge agrees on
  the leaders.
- **Leak check passed**: inputs are features only; no single feature is near-perfect for any event
  (strongest: `idio_vol_252`, |AUC − 0.5| = 0.29).
- **Training from 2012** instead of 2010 changes the mean log loss by −0.0014 to +0.0020 (mean
  +0.0003): no reason to leave the design check's choice of 2010.
- **Long windows** (8-K 5+ days after the release, 189–213 a year): without them the loss is
  0.001–0.002 lower; they are harder (other news in the window) but too few to matter.

#### The hypotheses [9b] (LightGBM, five seeds, every fold)

| Hypothesis | Change without the family | Expected sign | Verdict |
|---|---|---|---|
| H-R1 history | +0.0054 ± 0.0008 | 4 of 4 | supported |
| H-R2 volatility | +0.0024 ± 0.0001 | 4 of 4 | supported |
| H-R3 size and trading | +0.0008 ± 0.0002 | 4 of 4 | supported |
| H-R4 financials | +0.0034 ± 0.0008 | 4 of 4 | supported |
| H-R5 macro | +0.0020 ± 0.0011 | 3 of 4 | supported |
| H-R6 insider and short volume | −0.0001 ± 0.0002 | – | supported (adds nothing, as expected) |

H-R5 is the weakest: macro helps in three years and hurts in 2019, and the VIX's direction is
reversed in 2022. H-R1 also holds its second test: the baseline ranks the moves in every year.

#### The frozen model and the dry run [10]

- LightGBM with 15 leaves, at least 100 releases per leaf and 400 trees, chosen on the releases of
  2022; it learned from moves known through 30 December 2021; calibrations and band edges from the
  releases of 2022 (cutoffs 1 Jan 2022 and 1 Jan 2023). Fitted twice: identical (SC-002).
- **Dry run**: all 54,767 eligible 2023+ release eves and the newest rows of 4,979 companies still
  trading are scored; chances in [0, 1] and in order; a 2021 release is refused (SC-007). In the
  dry run `prev4_mean` is missing for everyone (sealed until the holdout test), so the model sees
  no history and few releases reach band 5; the holdout command supplies the history.
- Saved in MLflow (experiment `006-results-move`, run `frozen-lightgbm`), tagged approved (V2-55).

#### The approved model's code (V2-55)

- `src/sentinel/models/results_move.py` holds the approved model (LightGBM's forecast, the three
  calibrations, the bands, `fit`, `predict`), replacing Gemini's module; this notebook imports it
  for level 3, the checks and the frozen model, and gives the same numbers. Tests:
  `tests/unit/test_models_results_move.py`.
- `sentinel train results_move` builds the frozen model; `sentinel score results_move` scores the
  companies still trading (checked on 9 Nov: the same 4,979 companies and bands as section 10).
- The daily scores (paused, V2-42) store the chance above 10% as the forecast value, with the
  three chances and the band name; live outcomes compare it with a move above 10%.
- Before the daily scores resume (009): the approved models must be trained into `data/models/`
  (still Gemini's files of 2 Nov), and `prev4_mean` needs the moves after 2022, which the holdout
  test (010) computes.
- The top band's chance above 10% runs 2–5 points high: reported, not adjusted (a change now would
  be tuned on the test years); for the model card (008).